In [45]:
import os
usuario_home = os.path.expanduser("~")
print(usuario_home) 

C:\Users\x286384


In [46]:
import pandas as pd
import numpy as np
import os
#reemplazar con la ruta de los archivos correcta
#path_files="C:/Users/"+str(os.path.expanduser("~"))[-7:]+"/MerckGroup/Digitalización - DigitalProjectsLibrary/Planning/"
path_files="C:\\Users\\x286384\\MerckGroup\\Digitalización - Documents\\DigitalProjectsLibrary\\Planning\\"
FCST=pd.read_excel(path_files+"FCST Merck Abril 26.xlsx",sheet_name="Abril 2026")
PROD_DC_2=pd.read_excel(path_files+"PRODUCTOS DC.xlsx",sheet_name="Catalogo")

In [47]:
"""""
Module: P&G Forecast extraction 2
Purpose: extraer ## de jeringas para todos los meses por producto
Date: 30/07/2026
Author: J.Gonzalez
"""
logbalanceo=[]
#Columnas importantes
id_cols = ['SKUMERCK']

#Calculo de mes y año actual
Month_today=pd.to_datetime("today").month
Year_today=pd.to_datetime("today").year
demand_today=pd.to_datetime(str(Year_today) + "-01-" + str(Month_today), format="%Y-%m-%d")
demand_today=str(demand_today)[0:10]

#print(FCST.columns)
#FCST.columns = [pd.to_datetime(col, errors='coerce').strftime('%Y-%m') if pd.notna(pd.to_datetime(col, errors='coerce')) else col for col in FCST.columns]

matching_cols = []

for i in range(len(FCST.columns) - 1, -1, -1):  # from [-1] backwards
    col = FCST.columns[i]
    parsed = pd.to_datetime(col, errors='coerce')
    if pd.notna(parsed):
        matching_cols.append(col)
        if parsed.month <= Month_today and parsed.year <= Year_today:
            break  #
matching_cols = list(reversed(matching_cols))
#matching_cols = [pd.to_datetime(col, errors='coerce').strftime('%Y-%m') if pd.notna(pd.to_datetime(col, errors='coerce')) else col for col in matching_cols]
#Union de columnas id con columna demanda mes actual
#keep_cols = id_cols + matching_cols
FCST_month = FCST[id_cols + matching_cols].copy()

FCST_month = FCST_month.dropna(subset=['SKUMERCK']).reset_index(drop=True)

#Forecast de jeringas por mes actual en csv
FCST_month.to_csv("FCST_month.csv", index=False)

""""
#extraccion columnas de demanda del mes actual

#en caso de encontrar dos columnas con la misma fecha conserva la segunda que ya incluye el calculo total de piezas.
if len(matching_cols) >= 2:
    matching_cols = [matching_cols[-1]]

Union de columnas id con columna demanda mes actual
if matching_cols:
    keep_cols = id_cols + matching_cols
    FCST_month = FCST[keep_cols].copy()
else:
    FCST_month = FCST[id_cols].copy()
FCST_month = FCST_month.rename(columns={FCST_month.columns[-1]: 'Demanda'})
"""

'"\n#extraccion columnas de demanda del mes actual\n\n#en caso de encontrar dos columnas con la misma fecha conserva la segunda que ya incluye el calculo total de piezas.\nif len(matching_cols) >= 2:\n    matching_cols = [matching_cols[-1]]\n\nUnion de columnas id con columna demanda mes actual\nif matching_cols:\n    keep_cols = id_cols + matching_cols\n    FCST_month = FCST[keep_cols].copy()\nelse:\n    FCST_month = FCST[id_cols].copy()\nFCST_month = FCST_month.rename(columns={FCST_month.columns[-1]: \'Demanda\'})\n'

In [48]:
"""""
Module: Produccion por linea 2
Purpose: Separa los productos por familia, y despues por linea que utiliza dentro de cada mes esa familia especifica
Date: 30/07/2026
Author: J.Gonzalez
"""
PROD_DC_2 = PROD_DC_2.merge(FCST_month, on='SKUMERCK', how='left')
PROD_DC_2.drop(['SKUMERCK'], axis=1, inplace=True)
#PROD_DC_2_months = [col for col in PROD_DC_2.columns if col not in ['Nombre granel', 'Linea Granel 1', 'Linea Granel 2']]

#dicccionario para la suma del group
agg_dict = {col: 'sum' for col in PROD_DC_2.columns if col not in ['Nombre granel', 'Linea Granel 1', 'Linea Granel 2']}

#Group by
PROD_DC_2 = PROD_DC_2.groupby(['Nombre granel', 'Linea Granel 1', 'Linea Granel 2'], as_index=False).agg(agg_dict)

#Una columna para identificar DC
PROD_DC_2['Linea Granel 1'] = np.where(
    (PROD_DC_2['Linea Granel 1'] == 1) & (PROD_DC_2['Linea Granel 2'] == 1),
    'DC2',
    'DC1')
PROD_DC_2.drop(['Linea Granel 2'], axis=1, inplace=True)
PROD_DC_2 = PROD_DC_2.rename(columns={'Linea Granel 1': 'DC'})
PROD_DC_2.to_csv("PROD_DC_2.csv", index=False)
print(PROD_DC_2)


                              Nombre granel   DC  2026-07-28 00:00:00  \
0                      DCS NEUROBION 10,000  DC1             230400.0   
1                      DCS NEUROBION 10,000  DC2             230400.0   
2    DCS NEUROBION 10,000 DOUBLE FILTRATION  DC1                  0.0   
3                      DCS NEUROBION 25,000  DC1             125200.0   
4                    DEXABION DC BULK - MEX  DC1             115200.0   
5                    DEXABION DC BULK - MEX  DC2             230399.0   
6              DOLO NEUROBION DC BULK - MEX  DC1             115201.0   
7              DOLO NEUROBION DC BULK - MEX  DC2             728799.0   
8             DOLO NEUROBION FORTE DC - MEX  DC1             345600.0   
9             DOLO NEUROBION FORTE DC - MEX  DC2              77600.0   
10       DOLO NEUROBION FORTE DC BULK - MEX  DC1                  0.0   
11  SYR.NEUROBION 25000 DC BULK STEVA - GUA  DC1             220400.0   

    2026-08-29 00:00:00  2026-09-30 00:00:00  2026

In [50]:
"""""
Module: Balanceo mensual
Purpose: balancea DC2 mandando su residuo a DC1 para cada familia en cada mes 
Date: 30/07/2026
Author: J.Gonzalez
"""

jeringas=115200

familias= PROD_DC_2['Nombre granel'].unique()
familias_con_dc2= PROD_DC_2['Nombre granel'].duplicated
familias_con_dc2= familias_con_dc2.unique()

print(familias_con_dc2)


for familias in familias_con_dc2:
    for col in agg_dict.keys():
        if PROD_DC_2['DC'].loc[PROD_DC_2['DC'] == 'DC2'].values[0] % jeringas != 0:
            resdc2=PROD_DC_2['DC'].loc[PROD_DC_2['DC'] == 'DC2'].values[0]%jeringas
            PROD_DC_2['DC'].loc[PROD_DC_2['DC'] == 'DC1'].values[0]=PROD_DC_2['DC'].loc[PROD_DC_2['DC'] == 'DC1'].values[0]+(resdc2*jeringas)
            PROD_DC_2['DC'].loc[PROD_DC_2['DC'] == 'DC2'].values[0]=PROD_DC_2['DC'].loc[PROD_DC_2['DC'] == 'DC2'].values[0]-(resdc2*jeringas)

            

"""
for col in PROD_DC_2.columns:
repeat until PROD_DC_2[col]%jeringas==0:
if PROD_DC_2[col]%jeringas>0:
    PROD_DC_2[col]=PROD_DC_2[col]-jeringas
    PROD_DC_1[col]=PROD_DC_1[col]+jeringas
"""

#revision_residuo_dc1=PROD_DC_2[PROD_DC_2['DC'] == 'DC1'].copy()
#revision_residuo_dc1=revision_residuo_dc1.agg(agg_dict)/jeringas
#revision_residuo_dc2=PROD_DC_2[PROD_DC_2['DC'] == 'DC2'].copy()
#revision_residuo_dc2=revision_residuo_dc2.agg(agg_dict)/jeringas
#revision_residuo_total= PROD_DC_2.groupby(['Nombre granel'], as_index=False).agg(agg_dict)
#revision_residuo_total=revision_residuo_total.agg(agg_dict)/jeringas

#revision_residuo_dc2.to_csv("revision_residuo_dc2.csv", index=False)
#revision_residuo_dc1.to_csv("revision_residuo_dc1.csv", index=False)
#revision_residuo_total.to_csv("revision_residuo_total.csv", index=False)

#modulo balance


AttributeError: 'function' object has no attribute 'unique'

In [ ]:
"""""
Module: Balanceo sencillo
Purpose: reporta lotes minimos necesarios, y realiza un balanceo sencillo 
Date: 23/07/2026
Author: J.Gonzalez
"""
#Primer reporte es Generado como punto de comparación

# calculo de total de jeringas
TOTAL_DC1=DC1['Demanda'].sum()
TOTAL_DC2=DC2['Demanda'].sum()
TOTAL_DEMANDA=TOTAL_DC1+TOTAL_DC2

#calculo de lotes necesarios
LOTES_DC1=TOTAL_DC1/105200
LOTES_DC2=TOTAL_DC2/105200
lotes_necesarios=TOTAL_DEMANDA/105200

#calculo de residuos en jeringas
residuo_DC1=(1-(TOTAL_DC1/105200)%1)*105200
if residuo_DC1 == 105200.0:
    residuo_DC1 = 0
residuo_DC2=(1-(TOTAL_DC2/105200)%1)*105200
if residuo_DC2 == 105200.0:
    residuo_DC2=0
residuo_total=(1-(TOTAL_DEMANDA/105200)%1)*105200

# residuo directo de DC2 sin contar pedaceria de sobra
residuo_norm_dc2=((TOTAL_DC2/105200)%1)*105200


#generación de reporte 
print("Reporte situación actual")
print("TOTAL_DC1: "+str(TOTAL_DC1)+" total jeringas")
print("TOTAL_DC2: "+str(TOTAL_DC2)+" total jeringas")
print("TOTAL_DEMANDA: "+str(TOTAL_DEMANDA)+" total jeringas")
print("Lotes DC1 "+str(LOTES_DC1))
print("Lotes DC2 "+str(LOTES_DC2))
print("Total Demanda Lotes "+str(lotes_necesarios))
print("Residuo DC1 "+str(residuo_DC1)+" jeringas de sobra")
print("Residuo DC2 "+str(residuo_DC2)+" jeringas de sobra")
print("residuo de la demanda total "+str(residuo_DC1+residuo_DC2)+" jeringas de sobra")
print("Residuo Total "+str(residuo_total)+" jeringas de sobra")
print("")
print("")

"""
Aqui empieza el balanceo, solo busca ajustar para que la entrada de lotes a DC2 sea siempre un numero entero y mandar la pedaceria a DC1
"""
## busca el lote mas chico que pueda cumplir con las condiciones de ser menor que el residuo para balancear DC2
min_idx = DC2[(DC2['Demanda'] > residuo_norm_dc2) & (DC2['Units'] == 1)]['Demanda'].idxmin()
#reporte de linea antes de cambio
print("DC2:Fila antes de ajuste")
print(DC2.loc[[min_idx]])

# Se resta el residuo de la fila para balancear DC2
DC2.loc[min_idx, 'Demanda'] = DC2.loc[min_idx, 'Demanda'] - residuo_norm_dc2
print("DC2:Fila después de ajuste:")
print(DC2.loc[[min_idx]])

# Crea una nueva fila en DC1 con el residuo de DC2
row_to_copy = DC2.loc[[min_idx]]
row_to_copy['Demanda'] = residuo_norm_dc2
DC1 = pd.concat([DC1, row_to_copy], ignore_index=True)
print("DC1:Fila agregada:")
print(DC1.loc[[len(DC1)-1]])
print("")
#Log de cambios a DC1
if 'logbalanceo' not in globals():
    logbalanceo = []
else:
    logbalanceo.append(row_to_copy)

"""""
Genera el mismo reporte pero después del balanceo. 
"""

# calculo de total de jeringas
TOTAL_DC1=DC1['Demanda'].sum()
TOTAL_DC2=DC2['Demanda'].sum()
TOTAL_DEMANDA=TOTAL_DC1+TOTAL_DC2
#calculo de lotes necesarios
LOTES_DC1=TOTAL_DC1/105200
LOTES_DC2=TOTAL_DC2/105200
lotes_necesarios=TOTAL_DEMANDA/105200
#calculo de residuos en jeringas
residuo_DC1=(1-(TOTAL_DC1/105200)%1)*105200
if residuo_DC1 == 105200.0:
    residuo_DC1 = 0
residuo_DC2=(1-(TOTAL_DC2/105200)%1)*105200
if residuo_DC2 == 105200.0:
    residuo_DC2=0
residuo_total=(1-(TOTAL_DEMANDA/105200)%1)*105200

# residuo directo de DC2 sin contar pedaceria de sobra
residuo_norm_dc2=((TOTAL_DC2/105200)%1)*105200


#generación de reporte 
print("")
print("Reporte después del balanceo")
print("TOTAL_DC1: "+str(TOTAL_DC1)+" total jeringas")
print("TOTAL_DC2: "+str(TOTAL_DC2)+" total jeringas")
print("TOTAL_DEMANDA: "+str(TOTAL_DEMANDA)+" total jeringas")
print("Lotes DC1 "+str(LOTES_DC1))
print("Lotes DC2 "+str(LOTES_DC2))
print("Total Demanda Lotes "+str(lotes_necesarios))
print("Residuo DC1 "+str(residuo_DC1)+" jeringas de sobra")
print("Residuo DC2 "+str(residuo_DC2)+" jeringas de sobra")
print("residuo de la demanda total "+str(residuo_DC1+residuo_DC2)+" jeringas de sobra")
print("Residuo Total "+str(residuo_total)+" jeringas de sobra")

NameError: name 'DC1' is not defined